# Import Library

In [ ]:
import os
import pickle
import random
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from datetime import timedelta
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.autograd import Variable

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

print("Current directory: ", os.getcwd())

if not os.path.exists('./Results'):
    os.mkdir('./Results')
    
#### Create folders
if not os.path.exists('./Model'):
    os.mkdir('./Model')
    
#### Set seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(torch.cuda.is_available)
print(torch.version.cuda)
print(device)

def smooth_mae_loss(y_true, y_pred, delta=1.0):
    """
    Args:
        y_true: Ground truth, shape (batch_size, horizon_length).
        y_pred: Predictions, shape (batch_size, horizon_length).
        delta: Threshold to switch between MAE and MSE.
    Returns:
        Huber loss (scalar).
    """
    diff = y_true - y_pred
    loss = torch.where(torch.abs(diff) < delta,
                       0.5 * (diff ** 2),  # MSE region
                       delta * (torch.abs(diff) - 0.5 * delta))  # MAE region
    return loss.mean()  

#### model.eval() --> target = None
##### For create dataset
class TimeSeriesDataset(Dataset):
    def __init__(self, x, y, device):
        self.x = torch.tensor(x, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.float32).to(device)
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    
##### For create dataset
class TimeSeriesDataset_3(Dataset):
    def __init__(self, x, y, knw, device):
        self.x = torch.tensor(x, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.float32).to(device)
        self.knw = torch.tensor(knw, dtype=torch.float32).to(device)
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, index):
        return self.x[index], self.y[index], self.knw[index]
    

class QuantileLoss(nn.Module):
    def __init__(self, quantile):
        super(QuantileLoss, self).__init__()
        assert 0 < quantile < 1, "Quantile should be in (0, 1) range"
        self.quantile = quantile

    def forward(self, preds, target):
        errors = target - preds
        loss = torch.max((self.quantile - 1) * errors, self.quantile * errors)
        return loss.mean()

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_ts(y_test, y_pred,
            index,  # plotting period
            show=True,
            save=False,
            target=None,
            model=None,
            fct_h=None,
            seq_len=None):

    y_test = y_test.ravel()
    y_pred = y_pred.ravel()

    y_pred[y_pred < 0] = 0

    plt.figure(figsize=(10, 4))

    # Create a DataFrame to identify continuous segments
    df = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred}, index=index)

    # Find gaps in the index
    gaps = df.index.to_series().diff().dt.days.ne(1).cumsum()

    # Plot each continuous segment separately
    last_end = None
    for _, segment in df.groupby(gaps):
        if last_end is not None:
            plt.axvspan(last_end, segment.index[0], color='gray', alpha=0.3)
        plt.plot(segment.index, segment['y_test'], c='black', linewidth=1)
        plt.plot(segment.index, segment['y_pred'], c='red', linewidth=1)
        last_end = segment.index[-1]

    # Optional: scatter plot to highlight points
    plt.axhline(y=30, color='gray', linestyle='--', linewidth=2, alpha=0.5)
    plt.axhline(y=100, color='gray', linestyle='--', linewidth=2, alpha=0.5)

    plt.xlim(index[0], index[-1])
    plt.ylim(0 - y_test.max() * 0.05, max(y_test.max() * 1.05, y_pred.max() * 1.05))
    plt.yticks(size=13)
    plt.ylabel('Predicted Turbidity (NTU)', fontsize=13)
    plt.tight_layout()

    if save:
        folder_nam = "Model_evaluation"  # Modify this to your desired folder name

        import os

        if not os.path.exists(f'./Results/{folder_nam}'):
            os.mkdir(f'./Results/{folder_nam}')

        plt.savefig(f'./Results/{folder_nam}/{model}_{target}_{fct_h}_{seq_len}_temporal_variation.png', 
                    dpi=256, transparent=True)

    if show:
        plt.show()
    else:
        plt.close()
        
def confusion_matrix_file(df): ### TODO modifying code for weekly models
    n = len(df)

    new_df = pd.DataFrame()
    cols = ['30 to 100', '100 이상', 'Accuracy', 'Recall', '']

    for i in range(n):
        temp = df.iloc[i:i + 1].copy()
        temp.iloc[0, 0] = '0 to 30'
        new_df = pd.concat([new_df, temp], ignore_index=True)

        for j, value in enumerate(cols):
            new_row = [''] * len(df.columns)
            new_row[0] = value
            new_df = pd.concat([new_df, pd.DataFrame([new_row], columns=df.columns)], ignore_index=True)
        
        new_df.iloc[i * (len(cols) + 1), 4:9] = df.iloc[i, 12:17].values
        new_df.iloc[i * (len(cols) + 1) + 1, 1:4] = df.iloc[i, 4:7].values
        new_df.iloc[i * (len(cols) + 1) + 2, 1:4] = df.iloc[i, 7:10].values
        new_df.iloc[i * (len(cols) + 1) + 3, 1] = df.iloc[i, 10]
        new_df.iloc[i * (len(cols) + 1) + 4, 1] = df.iloc[i, 11]
        new_df.drop(new_df.columns[9:17], axis=1, inplace=True)
        
    new_df.columns = ['','0 to 30','30 to 100','100 이상', 'data', 'fct', 'site', 'model', 'Info']
    
    return new_df 

def confusion_tab_pred(y_, y_pred_,
                    fct,
                    model_nam,
                    info = None, ###
                    target = "NTU"):
    
    columns = [''] + [f'plot1_{i}_{j}' for i in range(3) for j in range(3)] + ['accuracy'] + ['recall'] + ['data']
    conf_save_temp = pd.DataFrame(columns=columns)
    
    accuracy, recall, conf = binning(y_.ravel(), y_pred_.ravel(), target)
    add_metrics_to_df(conf_save_temp, 0, pd.DataFrame(conf), accuracy, recall, 'te')

    conf_save_temp.loc[:, "fct"] = fct
    conf_save_temp.loc[:, "site"] = target
    conf_save_temp.loc[:, "model"] = model_nam
    conf_save_temp.loc[:, "Info"] = info

    return conf_save_temp

def calculate_score(y_test, y_pred, target):
    from sklearn.metrics import r2_score, mean_squared_error

    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2 = r2_score(y_test, y_pred)
    accuracy, recall, _ = binning(y_test, y_pred, target)

    return np.round(rmse, 4), np.round(r2, 4), np.round(accuracy, 4), np.round(recall, 4)

def binning(y_test, y_pred, target):
    from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
    
    if target in ['Synedra', 'ToxicCyano']: 
        cri = [-np.inf, 100, 1000, np.inf]
        cri2 = [-np.inf, 0.02, np.inf]
    if target in ['2MIB', 'Geosmin']: 
        cri = [-np.inf, 0.02, 0.05, np.inf]
        cri2 = [-np.inf, 0.02, np.inf]
        
    if "NTU" in target:
        cri = [-np.inf, 30, 100, np.inf]  # Criteria for binning
        cri2 = [-np.inf, 30, np.inf]  # Criteria for binning

    y_true_binned = pd.cut(y_test, bins=cri, labels=[0,1,2])
    y_pred_binned = pd.cut(y_pred, bins=cri, labels=[0,1,2])

    y_true_binned2 = pd.cut(y_test, bins=cri2, labels=[0,1])
    y_pred_binned2 = pd.cut(y_pred, bins=cri2, labels=[0,1])
    
    return accuracy_score(y_true_binned, y_pred_binned), recall_score(y_true_binned2, y_pred_binned2), confusion_matrix(y_true_binned, y_pred_binned, labels=[0, 1, 2])

def add_metrics_to_df(df, row_index, plot1, accuracy, recall, info):
    for i in range(3):
        for j in range(3):
            df.loc[row_index, f'plot1_{i}_{j}'] = plot1.iloc[i, j]
    df.loc[row_index, 'accuracy'] = accuracy
    df.loc[row_index, 'recall'] = recall
    df.loc[row_index, 'data'] = info

## Function for save R2, RMSE, accuracy and recall
def performance_tab_mod(y_te, y_pred_te,
                    fct,
                    model_nam,
                    info = None, ###
                    target = "NTU"):
    
    perf_save_temp = pd.DataFrame(columns = ['rmse', 'r2', 'accuracy', 'recall', 'fct', 'site', 'model', 'info'])
    
    perf_save_temp.loc[0, ['rmse', 'r2', 'accuracy', 'recall']] = calculate_score(y_te.ravel(), y_pred_te.ravel(), target)

    perf_save_temp.loc[0, 'data'] = 'te'

    perf_save_temp.loc[:, "fct"] = fct
    perf_save_temp.loc[:, "site"] = target
    perf_save_temp.loc[:, "model"] = model_nam
    perf_save_temp.loc[:, "info"] = info
    
    return perf_save_temp


 
def is_defined(variable_name): ## Utility function
    try:
        eval(variable_name)
        return True
    except NameError:
        return False
    
def create_sequences(data, target, knw_data, known,
                     seq_length, forecast_horizon, horizon_length, work_hour):
    xs = []
    ys = []
    knws = []
    
    for t in data[data.index.hour == work_hour].index:
        end_time = t
        start_time = end_time - pd.Timedelta(hours = seq_length - 1)
        
        x = data.loc[start_time:end_time, predictors].values
        if x.shape[0] == seq_length:
            xs.append(x)
            
            # x_ = data.loc[start_time:end_time, predictors]
            # if not is_defined('xs'):
            #     xs = np.expand_dims(x, axis = 0)
            # else:
            #     xs = np.concatenate((xs, np.expand_dims(x, axis = 0)))
    
    xs = np.array(xs)
    
    #### For target sequence
    sel_time = data[data.index.hour == work_hour][-xs.shape[0]:].index.floor('D')
    for t in sel_time:
        start_time = t + pd.Timedelta(days = forecast_horizon) 
        end_time = start_time + pd.Timedelta(days = horizon_length)

        y = data.loc[start_time:end_time][:-1]
        
        if y.shape[0] == horizon_length * 24:
            y.loc[:, 'date'] = y.index.date
            y_ = y.groupby('date')[target].mean().values
            ys.append(y_)
            
            knw = knw_data.loc[pd.to_datetime(knw_data['Date']) == t, known].values
            knws.append(knw)
            
            # if not is_defined('ys'):
            #     ys = np.expand_dims(y_, axis = 0)
            # else:        
            #     ys = np.concatenate((ys, np.expand_dims(y_, axis = 0)))
            
    ys = np.array(ys)
    knws = np.array(knws)/100
    xs = xs[:len(xs)-forecast_horizon-horizon_length, :, :]
    
    return xs, ys, knws

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0)  # Add batch dimension

    def forward(self, x):
        return x + self.encoding[:, :x.size(1), :].to(x.device)

class TransformerEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, nhead=8):
        super(TransformerEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=nhead)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.positional_encoding = PositionalEncoding(hidden_dim)

    def forward(self, input):
        input = self.input_projection(input)  # Project input to hidden_dim size
        input = self.positional_encoding(input)  # Apply positional encoding
        output = self.encoder(input.transpose(0, 1))  # Transformer expects seq_len first
        return output.transpose(0, 1)  # Return batch-first format

class TransformerDecoder(nn.Module):
    def __init__(self, hidden_dim, num_layers, output_dim, nhead=8):
        super(TransformerDecoder, self).__init__()
        self.output_projection = nn.Linear(hidden_dim, output_dim)
        decoder_layer = nn.TransformerDecoderLayer(d_model=hidden_dim, nhead=nhead)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.positional_encoding = PositionalEncoding(hidden_dim)

    def forward(self, input, encoder_output):
        input = self.positional_encoding(input)  # Apply positional encoding
        output = self.decoder(input.transpose(0, 1), encoder_output.transpose(0, 1))
        output = output.transpose(0, 1)
        return self.output_projection(output)


class TransformerSeq2Seq(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, forecast_horizon, nhead=8):
        super(TransformerSeq2Seq, self).__init__()
        self.encoder = TransformerEncoder(input_dim, hidden_dim, num_layers, nhead=nhead)
        self.decoder = TransformerDecoder(hidden_dim, num_layers, output_dim, nhead=nhead)
        self.forecast_horizon = forecast_horizon

    def forward(self, input, known_features):
        out_seq_len = self.forecast_horizon
        encoder_output = self.encoder(input)
        
        # 첫 번째 디코더 입력값으로 인코더 마지막 출력을 사용
        decoder_input = encoder_output[:, -1:, :]  # (batch_size, 1, hidden_dim)
        outputs = []

        for t in range(out_seq_len):
            # 디코더에서 예측된 현재 시점의 출력
            decoder_output = self.decoder(decoder_input, encoder_output)
            current_output = decoder_output[:, -1, :]  # (batch_size, hidden_dim)
            outputs.append(current_output)
            
            # 새로운 예측값을 디코더 입력값으로 설정
            decoder_input = current_output.unsqueeze(1)  # (batch_size, 1, hidden_dim)
        
        # 모든 시점의 예측값을 시퀀스 형태로 변환
        outputs = torch.stack(outputs, dim=1)  # (batch_size, out_seq_len, output_dim)
        return outputs
        
import optuna
from optuna.pruners import HyperbandPruner


def train_model(model, x_train, y_train, tr_known,
                loss_function, batch_size,
                optimizer,
                scheduler,
                num_epochs,
                min_, # For inverse transformation TODO change function using scaler function 
                max_,
                patience=200,
                n_splits=4):
    
    from sklearn.model_selection import KFold
    from torch.utils.data import DataLoader, Subset
    best_epoch = 0
    best_model = None
    best_loss = float('inf')
    tr_dataset = TimeSeriesDataset_3(x_train, y_train, tr_known, device)

    kf = KFold(n_splits=n_splits, shuffle=True)

    for fold, (train_idx, val_idx) in enumerate(kf.split(tr_dataset)):
        print(f"Fold {fold+1}/{n_splits}")
        best_fold_loss = float('inf')
        no_improve = 0

        train_subset = Subset(tr_dataset, train_idx)
        val_subset = Subset(tr_dataset, val_idx)

        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True) ### shuffle
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=True) ### shuffle


        for epoch in range(num_epochs):
            model.train()
            for x_batch, y_batch, knw_batch in train_loader:
                optimizer.zero_grad()
                y_pred = model(x_batch, knw_batch)
                
                # Inverse scaling
                y_pred = y_pred * (max_ - min_) + min_
                y_batch = y_batch * (max_ - min_) + min_
                
                y_pred = y_pred.squeeze(-1)
                loss = loss_function(y_pred, y_batch)
                loss.backward()
                optimizer.step()

            scheduler.step()
            # Validation step
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for x_val, y_val, knw_val in val_loader:
                    y_pred = model(x_val, knw_val)
                    
                    y_pred = y_pred * (max_ - min_) + min_
                    y_val = y_val * (max_ - min_) + min_
                    y_pred = y_pred.squeeze(-1) 
                    
                    val_loss += loss_function(y_pred, y_val).item()
                

            val_loss /= len(val_loader)

            print(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {val_loss:.4f}")

            if val_loss < best_fold_loss:
                best_fold_loss = val_loss
                best_fold_model = model.state_dict()
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}, best fold loss: {best_fold_loss:.4f}')
                break
        
        # Aggregate the best model and loss for the current fold
        if best_fold_loss < best_loss:
            best_loss = best_fold_loss
            best_epoch = epoch
            best_model = best_fold_model

    model.load_state_dict(best_model)
    return best_loss, best_epoch, model

def hyper_param_opt(model_name, target, 
                    x_train, y_train, tr_known,
                    min, max,
                    info, n_trials=50):
    
    study_name = f"{model_name}_{target}_{info}_optimization"
    
    # Create a study object with Hyperband Pruner
    pruner = HyperbandPruner()
    
    study = optuna.create_study(
        study_name=study_name,
        load_if_exists=True,
        direction='minimize',
        pruner=pruner
    )
    
    study.optimize(lambda trial: objective(trial, model_name,
                                           x_train, y_train, tr_known,
                                           min, max), n_trials=n_trials) ### TODO modifying min, max part

    print(f"Best trial for {model_name} on {target}:")
    trial = study.best_trial
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
        
    return study, trial, trial.user_attrs["Best_model"] ## For saving optimization results

##### objective function
def objective(trial, model_name,
              x_train, y_train, tr_known,
              min, max):
    known_feature_dim = 0
    # Define the hyperparameters to optimize
    hidden_dim = trial.suggest_categorical('hidden_dim', [16, 32, 48, 64, 80, 96, 104, 128])
    num_layers = trial.suggest_int('num_layers', 1, 5)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128, 256])
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)

    # Model configuration
    input_dim = x_train.shape[2]
    output_dim = 1
    horizon_length = 7
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ##
    model = TransformerSeq2Seq(input_dim, hidden_dim, output_dim, num_layers, forecast_horizon=horizon_length).to(device)
    
    # Training configuration
    loss_function = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=200, gamma=0.5)
    num_epochs = 100
   
    # Training loop
    vl_loss, best_epoch, best_model = train_model(model, x_train, y_train, tr_known,
                                                loss_function, batch_size, optimizer, scheduler, num_epochs, min, max)
    
    # trial.report(vl_loss, best_epoch)
    
    if trial.should_prune():
        raise optuna.TrialPruned()
    
    trial.set_user_attr("Best_model", best_model)
    
    return vl_loss

# Parameters

In [ ]:
# === DIRECTORY ===
os.chdir('/home/uoswem7/SaaS/논문용_모델/Turbidity')

# === HYPERPARAMETERS ===
sepa = "2023-01-01"
model_nam = "Transformer"
file_nam = 'NTU_Transformer'
targets =['PD1.NTU', 'PD2.NTU', 'PD3.NTU'] 
known_feature_dim = 0

hidden_dims = [16, 32, 64]
batch_size = [16, 32, 64, 128]
num_layer = [3, 5, 7, 9]
learning_rate = [0.001, 0.003, 0.005, 0.0005, 0.0001]

output_dim = 7
loss_nam = 'mse_loss'

if not os.path.exists('./Model/%s/%s'%(file_nam, loss_nam)):
    os.mkdir('./Model/%s/%s'%(file_nam, loss_nam))

# Modeling

In [ ]:
for target in targets: 
    print(target)
    
    seq_len = 36
    fct_h = 1
    h_len = 7

    def load_sequences(target):
        with open(f'./Model/{file_nam}/tr_xs_{target}.pkl', 'rb') as f:
            tr_xs = pickle.load(f)

        with open(f'./Model/{file_nam}/tr_ys_{target}.pkl', 'rb') as f:
            tr_ys = pickle.load(f)

        with open(f'./Model/{file_nam}/tr_knws_{target}.pkl', 'rb') as f:
            tr_knws = pickle.load(f)

        return tr_xs, tr_ys, tr_knws
    
    tr_xs, tr_ys, tr_knws = load_sequences(target)

    #### Remove samples with NaN
    tr_xs_ = tr_xs[~np.any(np.any(np.isnan(tr_xs), axis = 2), axis = 1)]
    tr_ys_ = tr_ys[~np.any(np.any(np.isnan(tr_xs), axis = 2), axis = 1)]
    tr_knws_ = tr_knws[~np.any(np.any(np.isnan(tr_xs), axis = 2), axis = 1)]

    tr_xs_ = tr_xs_[~np.isnan(tr_ys_).any(axis = 1)]
    tr_knws_ = tr_knws_[~np.isnan(tr_ys_).any(axis = 1)]
    tr_ys_ = tr_ys_[~np.isnan(tr_ys_).any(axis = 1)]

    print(tr_xs_.shape, tr_ys_.shape, tr_knws_.shape)

        ### Model implementation - test
    input_dim = tr_xs_.shape[2] ## Number of predictors
    horizon_length = h_len ### forecast_horizon
    #input_dim = x_train.shape[2]
    output_dim = 1
    horizon_length = 7
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    for hidden_dim in hidden_dims:
            for num_layers in num_layer:
                for bs in batch_size:
                    for lr in learning_rate:
                        print('hidden_dim:', hidden_dim)
                        print('num_layers:', num_layers)
                        print('batch_size:', bs)
                        print('learning_rate:', lr)
                        known_tr = torch.Tensor(tr_knws_)  ### test only


                        model = TransformerSeq2Seq(input_dim, hidden_dim, output_dim, num_layers, forecast_horizon=horizon_length).to(device)
    
                    

                        optimizer = optim.Adam(model.parameters(), lr=0.001)

                        min_, max_ = np.append(np.nanmin(tr_ys_, axis=(0, 1)), np.nanmin(tr_xs_, axis=(0, 1))),\
                                    np.append(np.nanmax(tr_ys_, axis=(0, 1)), np.nanmax(tr_xs_, axis=(0, 1)))
                        pickle.dump(np.array([min_, max_]), open(f'./Model/{file_nam}/{target}_Minmax.pkl', 'wb'))
                        
                        from torch.utils.data import DataLoader, Subset

                        tr_xs__, tr_ys__ = (tr_xs_ - min_[1:])/(max_[1:] - min_[1:]), (tr_ys_ - min_[0])/(max_[0] - min_[0])

                        tr_dataset = TimeSeriesDataset_3(tr_xs__, tr_ys__, known_tr, device)

                        tr_loader = DataLoader(tr_dataset, batch_size=bs, shuffle=True) ### shuffle

                        num_epochs = 100
                        for epoch in range(num_epochs): ### TODO add pbar
                            model.train()
                            for x_batch, y_batch, knw_batch in tr_loader:
                                optimizer.zero_grad()
                                y_pred = model(x_batch, knw_batch)
                                
                                #### Inverse scaling            
                                y_pred = y_pred * (max_[0] - min_[0]) + min_[0]
                                y_batch = y_batch * (max_[0] - min_[0]) + min_[0]
                                y_pred = y_pred.squeeze(-1)

                                if loss_nam == 'mae_loss':
                                    loss_function = nn.L1Loss()
                                    loss = loss_function(y_pred, y_batch)
                                elif loss_nam == 'mse_loss':
                                    loss_function = nn.MSELoss()
                                    loss = loss_function(y_pred, y_batch)
                                elif loss_nam == 'huber_loss':                                                               
                                    loss = smooth_mae_loss(y_batch, y_pred, 10)
                                
                                loss.backward()
                                optimizer.step()
                                    
                            print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {loss:.8f}")
                            

                        torch.save(model, './Model/%s/%s/%s_%s_bs_%s_hid_%s_num_%s_lr_%s.pt' % 
                                (file_nam, loss_nam ,model_nam, target, bs, hidden_dim, num_layers, lr))
                        
                        del model